In [7]:
#Instalar libreria
!pip install jinja2 weasyprint python-docx -q
print("Librerias instaladas")

Librerias instaladas


In [8]:
# Importar libreria
from pathlib import Path
print("Libreria importada")

# Carpeta base del proyecto
BASE = Path('/content/proyecto_smart_campus')

# Lista de todas las carpetas que necesitamos para Smart Campus
carpetas = [
    'datos/procesados',      # Datasets limpios e integrados (Guía 1 y 2)
    'datos/gobernanza',      # Dataset protegido (Guía 4)
    'datos/geo',             # Datasets geográficos y de sedes (Guía 6)
    'documentos',            # Documentos generales
    'documentos/geo',        # Documentos geográficos
    'documentos/reporte',    # Informes (Guía 7)
    'evidencias',            # Capturas generales
    'evidencias/geo',        # Capturas geográficas
    'evidencias/reporte',    # Gráficos del informe de campus
    'visualizaciones',       # Archivos de Power BI (Guía 5)
    'exportaciones',         # PDFs y DOCXs finales
    'templates',             # Plantillas Jinja2
    'scripts',               # Scripts del pipeline
]

# Crear cada carpeta
for c in carpetas:
    (BASE / c).mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada:")
print(f"Base: {BASE}")
print("\nCarpetas disponibles:")
for c in carpetas:
    print(f"   ├──  {c}/")

Libreria importada
Estructura de carpetas creada:
Base: /content/proyecto_smart_campus

Carpetas disponibles:
   ├──  datos/procesados/
   ├──  datos/gobernanza/
   ├──  datos/geo/
   ├──  documentos/
   ├──  documentos/geo/
   ├──  documentos/reporte/
   ├──  evidencias/
   ├──  evidencias/geo/
   ├──  evidencias/reporte/
   ├──  visualizaciones/
   ├──  exportaciones/
   ├──  templates/
   ├──  scripts/


In [9]:
# SUBIR DATASETS DE LA GUÍA 1 Y 2
from google.colab import files

print("Sube los datasets de Smart Campus:")
print("   - dataset_integrado_Smart_Campus.csv")
print("   - dataset_limpio_Smart_Campus.csv")
print("Presiona 'Choose Files' y selecciona ambos archivos\n")

uploaded = files.upload()

for nombre, contenido in uploaded.items():
    destino = BASE / 'datos' / 'procesados' / nombre
    destino.write_bytes(contenido)
    print(f"  {nombre} → datos/procesados/")

# SUBIR DATASET DE LA GUÍA 4 (gobernanza)
print("Sube el dataset de la Guía 4:")
print("   - dataset_publicable_Smart_Campus.csv")
print("Presiona 'Choose Files'\n")

uploaded = files.upload()

for nombre, contenido in uploaded.items():
    destino = BASE / 'datos' / 'gobernanza' / nombre
    destino.write_bytes(contenido)
    print(f"    {nombre} → datos/gobernanza/")

# SUBIR DATASETS DE LA GUÍA 6 (geo)
print("Sube los datasets geográficos de la Guía 6:")
print("   - smartcampus_nacional_arcgis.csv")
print("Presiona 'Choose Files'\n")

uploaded = files.upload()

for nombre, contenido in uploaded.items():
    destino = BASE / 'datos' / 'geo' / nombre
    destino.write_bytes(contenido)
    print(f"    {nombre} → datos/geo/")

Sube los datasets de Smart Campus:
   - dataset_integrado_Smart_Campus.csv
   - dataset_limpio_Smart_Campus.csv
Presiona 'Choose Files' y selecciona ambos archivos



Saving dataset_limpio_Smart_Campus.csv to dataset_limpio_Smart_Campus (1).csv
Saving dataset_integrado_Smart_Campus.csv to dataset_integrado_Smart_Campus (1).csv
  dataset_limpio_Smart_Campus (1).csv → datos/procesados/
  dataset_integrado_Smart_Campus (1).csv → datos/procesados/
Sube el dataset de la Guía 4:
   - dataset_publicable_Smart_Campus.csv
Presiona 'Choose Files'



Saving dataset_publicable_Smart_Campus.csv to dataset_publicable_Smart_Campus (1).csv
    dataset_publicable_Smart_Campus (1).csv → datos/gobernanza/
Sube los datasets geográficos de la Guía 6:
   - smartcampus_nacional_arcgis.csv
Presiona 'Choose Files'



Saving smartcampus_nacional_arcgis.csv to smartcampus_nacional_arcgis.csv
    smartcampus_nacional_arcgis.csv → datos/geo/


In [10]:
# VERIFICAR LOS DATASETS SUBIDOS
archivos = {
    'Guía 1 & 2 (Integrado)': BASE / 'datos' / 'procesados' / 'dataset_integrado_Smart_Campus.csv',
    'Guía 2 (Limpio)':        BASE / 'datos' / 'procesados' / 'dataset_limpio_Smart_Campus.csv',
    'Guía 4 (Gobernanza)':    BASE / 'datos' / 'gobernanza' / 'dataset_publicable_Smart_Campus.csv',
    'Guía 6 (Geo ArcGIS)':    BASE / 'datos' / 'geo' / 'smartcampus_nacional_arcgis.csv',
}

print("VERIFICANDO DATASETS SUBIDOS:\n")
todos_ok = True
for nombre, ruta in archivos.items():
    if ruta.exists():
        tamaño = ruta.stat().st_size
        print(f" OK  {nombre}: {ruta.name} ({tamaño:,} bytes)")
    else:
        print(f"  FAILED {nombre}: NO ENCONTRADO")
        print(f"      Ruta: {ruta}")
        todos_ok = False

print("\n" + "=" * 60)
if todos_ok:
    print("Good job ¡Todos los datasets de Smart Campus están listos para usar!")
else:
    print("Warning!! Faltan archivos. Súbelos y vuelve a ejecutar esta celda.")

VERIFICANDO DATASETS SUBIDOS:

  FAILED Guía 1 & 2 (Integrado): NO ENCONTRADO
      Ruta: /content/proyecto_smart_campus/datos/procesados/dataset_integrado_Smart_Campus.csv
 OK  Guía 2 (Limpio): dataset_limpio_Smart_Campus.csv (1,398,972 bytes)
  FAILED Guía 4 (Gobernanza): NO ENCONTRADO
      Ruta: /content/proyecto_smart_campus/datos/gobernanza/dataset_publicable_Smart_Campus.csv
 OK  Guía 6 (Geo ArcGIS): smartcampus_nacional_arcgis.csv (2,770 bytes)

Warning!! Faltan archivos. Súbelos y vuelve a ejecutar esta celda.


In [11]:
# IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from jinja2 import Template
import base64
import json
import warnings
warnings.filterwarnings('ignore')  # Ignorar warnings menores

# CONFIGURAR MATPLOTLIB
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print(" Librerías importadas")
print(f"    - pandas: {pd.__version__}")
print(f"    - matplotlib: {plt.matplotlib.__version__}")

 Librerías importadas
    - pandas: 2.2.3
    - matplotlib: 3.10.0


In [12]:
# CARGAR DATASETS
df = pd.read_csv(BASE / 'datos' / 'procesados' / 'dataset_limpio_Smart_Campus.csv')
df['fecha'] = pd.to_datetime(df['fecha'])

ruta_geo = BASE / 'datos' / 'geo' / 'smartcampus_nacional_arcgis.csv'
if not ruta_geo.exists():
    for archivo_alt in (BASE / 'datos' / 'geo').glob('smartcampus_nacional_arcgis*.csv'):
        if archivo_alt.name != 'smartcampus_nacional_arcgis.csv':
            archivo_alt.rename(ruta_geo)
            print(f" Se renombró {archivo_alt.name} a {ruta_geo.name} para corregir la ruta.")
            break

df_geo = pd.read_csv(ruta_geo)

print(f"\nDatasets cargados:")
print(f"   - Asistencia Campus: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"   - Edificios/Geo: {df_geo.shape[0]} filas")

print(f"\nPrimeras 3 filas del dataset principal:")
print(df.head(3).to_string())

# CALCULAR KPIs DE SMART CAMPUS
kpis = {
    'total_registros':       df['id_registro'].nunique(),
    'estudiantes_totales':   df['estudiantes_presentes'].sum(),
    'horas_estudiante':      df['horas_estudiante_totales'].sum(),
    'ocupacion_promedio':    df['porcentaje_ocupacion'].mean(),
    'capacidad_max_media':   df['capacidad_maxima'].mean(),
    'fecha_inicio':          df['fecha'].min().strftime('%d/%m/%Y'),
    'fecha_fin':             df['fecha'].max().strftime('%d/%m/%Y'),
    'periodo':               f"{df['fecha'].min().strftime('%d/%m/%Y')} - {df['fecha'].max().strftime('%d/%m/%Y')}"
}

print("KPIs DEL SMART CAMPUS")
print("=" * 50)
print(f"Total Registros:         {kpis['total_registros']:>10}")
print(f"Estudiantes Totales:     {kpis['estudiantes_totales']:>10,}")
print(f"Horas-Estudiante:        {kpis['horas_estudiante']:>10,.2f}")
print(f"Ocupación Promedio:      {kpis['ocupacion_promedio']:>9.1f}%")
print(f"Capacidad Máx Promedio:  {kpis['capacidad_max_media']:>10.1f}")
print(f"Periodo:                 {kpis['periodo']}")
print("=" * 50)

# ANÁLISIS POR FACULTAD
df_facultad = df.groupby('facultad').agg(
    sesiones=('id_registro', 'count'),
    estudiantes=('estudiantes_presentes', 'sum'),
    horas_totales=('horas_estudiante_totales', 'sum'),
    ocupacion_prom=('porcentaje_ocupacion', 'mean')
).reset_index()
df_facultad = df_facultad.sort_values('horas_totales', ascending=False)

print("\nANÁLISIS POR FACULTAD:")
print(df_facultad.to_string(index=False))

# ANÁLISIS POR TIPO DE RECURSO
df_recurso = df.groupby('tipo_recurso').agg(
    sesiones=('id_registro', 'count'),
    horas_totales=('horas_estudiante_totales', 'sum'),
    ocupacion_prom=('porcentaje_ocupacion', 'mean')
).reset_index()
df_recurso = df_recurso.sort_values('horas_totales', ascending=False)

print("\nANÁLISIS POR TIPO DE RECURSO:")
print(df_recurso.to_string(index=False))

# ANÁLISIS POR SEDE (Guía 6)
df_sede = df_geo.groupby('sede').agg(
    total_estudiantes=('total_estudiantes', 'sum'),
    ocupacion_promedio=('ocupacion_promedio', 'mean'),
    edificios=('edificio', 'count')
).reset_index()
df_sede = df_sede.sort_values('total_estudiantes', ascending=False)

print("\nANÁLISIS POR SEDE (GEO):")
print(df_sede.to_string(index=False))


Datasets cargados:
   - Asistencia Campus: 10000 filas × 17 columnas
   - Edificios/Geo: 20 filas

Primeras 3 filas del dataset principal:
   id_registro      fecha  id_recurso hora_inicio hora_fin  estudiantes_presentes    facultad  horas_uso  horas_estudiante_totales         sede       nombre_recurso      edificio tipo_recurso  capacidad_maxima  porcentaje_ocupacion     estado prioridad
0            1 2026-06-28         101       10:00    13:00                     24  Ingenieria        3.0                      72.0  Santa Tecla      Lab Redes Cisco    Edificio B  Laboratorio                35                 68.57  Operativo   Ninguna
1            2 2026-02-05         101       09:00    12:00                     10  Ingenieria        3.0                      30.0  Santa Tecla      Lab Redes Cisco    Edificio B  Laboratorio                35                 28.57  Operativo   Ninguna
2            3 2026-03-23         302       13:00    15:00                     31  Ingenieria        

In [13]:
# ===================================================================
# GRÁFICO 1: Tendencia temporal de ocupación (líneas)
# ===================================================================
df_tendencia = df.groupby('fecha')['porcentaje_ocupacion'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_tendencia['fecha'], df_tendencia['porcentaje_ocupacion'],
        marker='o', linewidth=2.5, markersize=6,
        color='#2c7bb6', markerfacecolor='#1a5490')
ax.fill_between(df_tendencia['fecha'], df_tendencia['porcentaje_ocupacion'], alpha=0.2, color='#2c7bb6')
ax.set_title('Evolución de Ocupación Promedio por Fecha', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Ocupación Promedio (%)', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
ruta_tendencia = BASE / 'evidencias' / 'reporte' / 'grafico_tendencia.png'
plt.savefig(ruta_tendencia, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 1: {ruta_tendencia.name}")

# ===================================================================
# GRÁFICO 2: Uso por facultad (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Blues(np.linspace(0.4, 0.9, len(df_facultad)))
barras = ax.bar(df_facultad['facultad'], df_facultad['horas_totales'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)
ax.set_title('Horas-Estudiante Totales por Facultad', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Facultad', fontsize=11)
ax.set_ylabel('Horas-Estudiante', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_facultad['horas_totales']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f} hrs', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_facultad = BASE / 'evidencias' / 'reporte' / 'grafico_facultades.png'
plt.savefig(ruta_facultad, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 2: {ruta_facultad.name}")

# ===================================================================
# GRÁFICO 3: Uso por tipo de recurso (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = ['#e74c3c', '#3498db', '#2ecc71']
barras = ax.bar(df_recurso['tipo_recurso'], df_recurso['horas_totales'],
                color=colores, edgecolor='#2c3e50', linewidth=1.5)
ax.set_title('Uso por Tipo de Recurso', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tipo de Recurso', fontsize=11)
ax.set_ylabel('Horas-Estudiante', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_recurso['horas_totales']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:,.0f} hrs', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_recurso = BASE / 'evidencias' / 'reporte' / 'grafico_recurso.png'
plt.savefig(ruta_recurso, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 3: {ruta_recurso.name}")

# ===================================================================
# GRÁFICO 4: Ocupación por Sede (barras)
# ===================================================================
fig, ax = plt.subplots(figsize=(10, 6))
colores = plt.cm.Greens(np.linspace(0.4, 0.9, len(df_sede)))
barras = ax.bar(df_sede['sede'], df_sede['ocupacion_promedio'],
                color=colores, edgecolor='#1a5490', linewidth=1.5)
ax.set_title('Ocupación Promedio por Sede (Análisis Geoespacial)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Sede', fontsize=11)
ax.set_ylabel('Ocupación Promedio (%)', fontsize=11)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

for barra, valor in zip(barras, df_sede['ocupacion_promedio']):
    height = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., height,
            f'{valor:.1f}%', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

plt.tight_layout()
ruta_sede = BASE / 'evidencias' / 'reporte' / 'grafico_sede.png'
plt.savefig(ruta_sede, dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
print(f"------ Gráfico 4: {ruta_sede.name}")

print("\n¡Los 4 gráficos de Smart Campus se generaron correctamente!")

# VERIFICAR LOS 4 GRÁFICOS
graficos = {
    'Tendencia':  ruta_tendencia,
    'Facultades': ruta_facultad,
    'Recurso':    ruta_recurso,
    'Sede':       ruta_sede,
}

print("VERIFICANDO GRÁFICOS:\n")
for nombre, ruta in graficos.items():
    if ruta.exists():
        tamaño = ruta.stat().st_size / 1024
        print(f"OK {nombre}: {ruta.name} ({tamaño:.1f} KB)")
    else:
        print(f"FAILED {nombre}: NO ENCONTRADO")

------ Gráfico 1: grafico_tendencia.png
------ Gráfico 2: grafico_facultades.png
------ Gráfico 3: grafico_recurso.png
------ Gráfico 4: grafico_sede.png

¡Los 4 gráficos de Smart Campus se generaron correctamente!
VERIFICANDO GRÁFICOS:

OK Tendencia: grafico_tendencia.png (91.8 KB)
OK Facultades: grafico_facultades.png (50.6 KB)
OK Recurso: grafico_recurso.png (53.3 KB)
OK Sede: grafico_sede.png (58.1 KB)


In [14]:
# INSTALAR HARFBUZZ-SUBSET
!apt-get update -qq
!apt-get install -y libharfbuzz-subset0 -qq

# Reinstalar WeasyPrint con soporte para HarfBuzz
!pip install -U weasyprint -q

print("HarfBuzz-Subset instalado")

# TEMPLATE HTML CON JINJA2 PARA SMART CAMPUS
template_informe = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Informe Gerencial - Smart Campus Analytics</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; color: #333; line-height: 1.6; }
        h1 { color: #1a5490; border-bottom: 3px solid #1a5490; padding-bottom: 10px; }
        h2 { color: #2c7bb6; margin-top: 30px; border-bottom: 1px solid #ddd; padding-bottom: 5px; }
        h3 { color: #444; margin-top: 20px; }
        .portada { text-align: center; margin-bottom: 40px; padding: 25px; background: #f0f7ff; border-radius: 8px; }
        .portada h1 { border: none; font-size: 28px; margin-bottom: 5px; }
        .portada h2 { border: none; color: #666; margin-top: 5px; }
        .kpi-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin: 20px 0; }
        .kpi-card { background: #f0f7ff; padding: 15px; border-left: 4px solid #2c7bb6; border-radius: 4px; }
        .kpi-valor { font-size: 24px; font-weight: bold; color: #1a5490; }
        .kpi-label { font-size: 12px; color: #666; text-transform: uppercase; margin-bottom: 5px; }
        .hallazgo { background: #f9f9f9; padding: 15px; border-left: 4px solid #ff9800; margin: 15px 0; border-radius: 4px; }
        .hallazgo h3 { margin-top: 0; color: #e65100; }
        .limitacion { background: #fff3cd; padding: 10px; border-left: 4px solid #ffc107; margin: 10px 0; border-radius: 4px; }
        .limitacion-titulo { font-weight: bold; color: #856404; margin-right: 5px; }
        table { border-collapse: collapse; width: 100%; margin: 15px 0; }
        th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
        th { background: #2c7bb6; color: white; }
        tr:nth-child(even) { background: #f9f9f9; }
        .pie { font-size: 11px; color: #999; margin-top: 40px; border-top: 1px solid #ddd; padding-top: 10px; text-align: center; }
        img { max-width: 100%; margin: 10px 0; border: 1px solid #eee; border-radius: 4px; }
        .metodologia { background: #e8f5e9; padding: 15px; border-radius: 4px; margin: 15px 0; }
    </style>
</head>
<body>

<div class="portada">
    <h1>INFORME GERENCIAL - SMART CAMPUS</h1>
    <h2>Smart Campus Analytics</h2>
    <p><strong>Periodo:</strong> {{ periodo }}</p>
    <p><strong>Fecha de generación:</strong> {{ fecha_generacion }}</p>
    <p><strong>Versión:</strong> {{ version }}</p>
    <p><strong>Equipo:</strong> {{ equipo }}</p>
</div>

<h2>1. Resumen Ejecutivo</h2>
<p>El presente informe analiza la ocupación de laboratorios y recursos tecnológicos en Smart Campus durante el periodo {{ periodo }}.
Su propósito es apoyar la toma de decisiones para la optimización de espacios y la distribución eficiente en las distintas sedes,
a partir de la evidencia de los datasets integrados (Guía 1 y 2), gobernanza (Guía 4) y análisis territorial en ArcGIS (Guía 6).</p>

<h3>Indicadores Principales</h3>
<div class="kpi-grid">
    <div class="kpi-card">
        <div class="kpi-label">Total Registros</div>
        <div class="kpi-valor">{{ kpis.total_registros }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Estudiantes Atendidos</div>
        <div class="kpi-valor">{{ "{:,}".format(kpis.estudiantes_totales) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Horas-Estudiante</div>
        <div class="kpi-valor">{{ "%.1f"|format(kpis.horas_estudiante) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Ocupación Promedio</div>
        <div class="kpi-valor">{{ "%.1f"|format(kpis.ocupacion_promedio) }}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Capacidad Máx Promedio</div>
        <div class="kpi-valor">{{ "%.1f"|format(kpis.capacidad_max_media) }}</div>
    </div>
    <div class="kpi-label" style="display:none;"></div>
</div>

<h2>2. Contexto y Metodología</h2>
<div class="metodologia">
    <p><strong>Fuentes utilizadas:</strong> Dataset integrado Smart Campus, Dataset limpio, Dataset publicable (Gobernanza), Capas ArcGIS (Sedes).</p>
    <p><strong>Proceso:</strong> Ingesta → Limpieza → Integración → Enmascaramiento/Gobernanza → Análisis geoespacial.</p>
</div>

<h2>3. Resultados Generales</h2>

<h3>3.1 Evolución de Ocupación</h3>
<img src="{{ ruta_tendencia }}" alt="Tendencia de ocupación">
<p><strong>Lectura:</strong> La ocupación de los espacios muestra un comportamiento dinámico a lo largo de las fechas registradas.</p>

<h3>3.2 Uso por Facultad</h3>
<img src="{{ ruta_facultad }}" alt="Uso por facultad">
<p><strong>Lectura:</strong> La facultad de {{ facultad_top }} concentra el mayor volumen de uso con {{ "{:,.1f}".format(horas_facultad_top) }} horas-estudiante.</p>

<h3>3.3 Uso por Tipo de Recurso</h3>
<img src="{{ ruta_recurso }}" alt="Uso por recurso">
<p><strong>Lectura:</strong> El recurso tipo {{ recurso_top }} presenta un alto nivel de demanda institucional.</p>

<h2>4. Análisis Geoespacial</h2>
<img src="{{ ruta_sede }}" alt="Ocupación por sede">
<p><strong>Patrón observado:</strong> La sede de {{ sede_top }} registra los indicadores más altos de ocupación promedio ({{ "%.1f"|format(ocupacion_sede_top) }}%).</p>
<p><strong>Lectura responsable:</strong> Existe concentración de asistencia en sedes principales; se recomienda escalamiento de recursos.</p>

<h2>5. Hallazgos Principales</h2>
{% for h in hallazgos %}
<div class="hallazgo">
    <h3>{{ loop.index }}. {{ h.titulo }}</h3>
    <p><strong>Evidencia:</strong> {{ h.evidencia }}</p>
    <p><strong>Interpretación:</strong> {{ h.interpretacion }}</p>
    <p><strong>Impacto:</strong> {{ h.impacto }}</p>
    <p><strong>Decisión sugerida:</strong> {{ h.decision }}</p>
    <p><em>Limitación: {{ h.limitacion }}</em></p>
</div>
{% endfor %}

<h2>6. Recomendaciones</h2>
<table>
    <thead>
        <tr>
            <th>#</th>
            <th>Acción</th>
            <th>Prioridad</th>
            <th>Responsable</th>
            <th>Plazo</th>
        </tr>
    </thead>
    <tbody>
        {% for r in recomendaciones %}
        <tr>
            <td>{{ loop.index }}</td>
            <td>{{ r.accion }}</td>
            <td>{{ r.prioridad }}</td>
            <td>{{ r.responsable }}</td>
            <td>{{ r.plazo }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>

<h2>7. Limitaciones del Análisis</h2>
{% for l in limitaciones %}
<div class="limitacion"><span class="limitacion-titulo">[!]</span> {{ l }}</div>
{% endfor %}

<h2>8. Conclusiones</h2>
<p>El análisis del periodo {{ periodo }} muestra una ocupación promedio general del {{ "%.1f"|format(kpis.ocupacion_promedio) }}%.
La facultad de {{ facultad_top }} y la sede de {{ sede_top }} lideran la utilización de la infraestructura tecnológica y de laboratorios.</p>

<div class="pie">
    <p>Informe generado automáticamente por el pipeline de reportería - Smart Campus Analytics</p>
    <p>Proyecto integrador - Guía Práctica de Reportería Automatizada</p>
</div>

</body>
</html>
"""

print("Template Jinja2 definido para Smart Campus")

# FUNCIÓN PARA CONVERTIR IMAGEN A BASE64
def imagen_a_base64(ruta):
    """Convierte una imagen a base64 para incrustarla directamente en HTML."""
    with open(ruta, 'rb') as f:
        return f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

# PREPARAR DATOS PARA EL TEMPLATE
datos_template = {
    'periodo': kpis['periodo'],
    'fecha_generacion': datetime.now().strftime('%d/%m/%Y %H:%M'),
    'version': '1.0',
    'equipo': 'Equipo Smart Campus Analytics',
    'kpis': type('KPIs', (), kpis)(),

    # Imágenes en base64
    'ruta_tendencia': imagen_a_base64(ruta_tendencia),
    'ruta_facultad': imagen_a_base64(ruta_facultad),
    'ruta_recurso': imagen_a_base64(ruta_recurso),
    'ruta_sede': imagen_a_base64(ruta_sede),

    # Datos para texto
    'facultad_top': df_facultad.iloc[0]['facultad'],
    'horas_facultad_top': df_facultad.iloc[0]['horas_totales'],
    'recurso_top': df_recurso.iloc[0]['tipo_recurso'],
    'sede_top': df_sede.iloc[0]['sede'],
    'ocupacion_sede_top': df_sede.iloc[0]['ocupacion_promedio'],

    # Hallazgos
    'hallazgos': [
        {
            'titulo': f'Alta demanda en la facultad de {df_facultad.iloc[0]["facultad"]}',
            'evidencia': f'La facultad acumuló {df_facultad.iloc[0]["horas_totales"]:,.0f} horas-estudiante en el periodo.',
            'interpretacion': 'Demanda sostenida de recursos académicos y de laboratorios.',
            'impacto': 'Mayor desgaste de equipos y alta concurrencia en franjas pico.',
            'decision': 'Optimizar turnos de uso y mantenimiento preventivo.',
            'limitacion': 'El análisis no incluye costos de mantenimiento operativo.'
        },
        {
            'titulo': f'Concentración de uso en laboratorios ({df_recurso.iloc[0]["tipo_recurso"]})',
            'evidencia': f'El recurso tipo {df_recurso.iloc[0]["tipo_recurso"]} lidera las horas de utilización.',
            'interpretacion': 'Los estudiantes requieren espacios físicos especializados.',
            'impacto': 'Saturación en determinados horarios de la semana.',
            'decision': 'Evaluar ampliación de capacidad o habilitación de salas adicionales.',
            'limitacion': 'Datos basados en registros históricos del campus.'
        },
        {
            'titulo': f'Importancia estratégica de la sede {df_sede.iloc[0]["sede"]}',
            'evidencia': f'La sede registra una ocupación promedio del {df_sede.iloc[0]["ocupacion_promedio"]:.1f}%.',
            'interpretacion': 'Foco principal de actividad académica y tecnológica.',
            'impacto': 'Demanda constante de conectividad e infraestructura.',
            'decision': 'Priorizar inversiones de mejora tecnológica en esta sede.',
            'limitacion': 'Correlación geográfica sujeta a la densidad de edificios.'
        }
    ],

    # Recomendaciones
    'recomendaciones': [
        {'accion': f'Monitorear semanalmente la ocupación en {df_facultad.iloc[0]["facultad"]}', 'prioridad': 'Alta', 'responsable': 'Coordinación Académica', 'plazo': 'Semanal'},
        {'accion': f'Revisar capacidad de soporte técnico para {df_recurso.iloc[0]["tipo_recurso"]}', 'prioridad': 'Alta', 'responsable': 'Jefatura de TI', 'plazo': 'Mensual'},
        {'accion': 'Optimizar la distribución horaria para evitar sobrecupos', 'prioridad': 'Media', 'responsable': 'Administración de Campus', 'plazo': 'Trimestral'},
        {'accion': 'Validar tendencias de crecimiento con analítica predictiva', 'prioridad': 'Media', 'responsable': 'Analista de Datos', 'plazo': 'Próximo periodo'},
        {'accion': f'Estudiar oportunidades de expansión en sede {df_sede.iloc[0]["sede"]}', 'prioridad': 'Alta', 'responsable': 'Dirección de Operaciones', 'plazo': 'Trimestral'}
    ],

    # Limitaciones
    'limitaciones': [
        'Volumen de datos correspondiente al conjunto de registros simulados y validados del campus.',
        'Periodo limitado a las fechas analizadas sin estacionalidad interanual completa.',
        'Datos orientados a demostrar el pipeline técnico de Smart Campus.',
        'El análisis no contempla variables externas como condiciones climáticas o eventos especiales.',
        'La correlación geoespacial indica asociación y no causalidad directa.'
    ]
}

# Renderizar template
template = Template(template_informe)
html_renderizado = template.render(**datos_template)

# Guardar HTML
ruta_html = BASE / 'exportaciones' / 'informe_smart_campus.html'
ruta_html.write_text(html_renderizado, encoding='utf-8')
print(f"HTML generado: {ruta_html.name}")

# Convertir a PDF
from weasyprint import HTML
ruta_pdf = BASE / 'exportaciones' / 'informe_smart_campus_v1.0.pdf'
HTML(string=html_renderizado).write_pdf(str(ruta_pdf))
print(f"PDF generado: {ruta_pdf.name}")
print(f"Tamaño: {ruta_pdf.stat().st_size / 1024:.1f} KB")

# Descargar PDF
from google.colab import files
files.download(str(ruta_pdf))
print("PDF descargado")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libharfbuzz-subset0:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../libharfbuzz-subset0_8.3.0-2build2_amd64.deb ...
Unpacking libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Setting up libharfbuzz-subset0:amd64 (8.3.0-2build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PDF descargado


In [15]:
# GENERAR DOCX CON PYTHON-DOCX
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()

# Título
titulo = doc.add_heading('INFORME GERENCIAL - SMART CAMPUS', 0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

subtitulo = doc.add_heading('Smart Campus Analytics', level=1)
subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(f'Periodo: {kpis["periodo"]}')
doc.add_paragraph(f'Fecha de generación: {datetime.now().strftime("%d/%m/%Y %H:%M")}')
doc.add_paragraph(f'Versión: 1.0')

# Resumen ejecutivo
doc.add_heading('1. Resumen Ejecutivo', level=2)
doc.add_paragraph(f'El presente informe analiza la gestión y ocupación de espacios en Smart Campus durante el periodo {kpis["periodo"]}.')

# KPIs
doc.add_heading('Indicadores Principales', level=3)
tabla_kpis = doc.add_table(rows=5, cols=2)
tabla_kpis.style = 'Light Grid Accent 1'

datos_kpis = [
    ('Total Registros', f'{kpis["total_registros"]}'),
    ('Estudiantes Atendidos', f'{kpis["estudiantes_totales"]:,}'),
    ('Horas-Estudiante', f'{kpis["horas_estudiante"]:,.2f}'),
    ('Ocupación Promedio', f'{kpis["ocupacion_promedio"]:.1f}%'),
    ('Capacidad Máx Promedio', f'{kpis["capacidad_max_media"]:.1f}')
]

for i, (label, valor) in enumerate(datos_kpis):
    tabla_kpis.rows[i].cells[0].text = label
    tabla_kpis.rows[i].cells[1].text = valor

# Resultados con gráficos
doc.add_heading('2. Resultados Generales', level=2)
doc.add_heading('2.1 Evolución de Ocupación', level=3)
doc.add_picture(str(ruta_tendencia), width=Inches(6))

doc.add_heading('2.2 Uso por Facultad', level=3)
doc.add_picture(str(ruta_facultad), width=Inches(6))

doc.add_heading('2.3 Uso por Tipo de Recurso', level=3)
doc.add_picture(str(ruta_recurso), width=Inches(6))

doc.add_heading('2.4 Análisis Geoespacial por Sede', level=3)
doc.add_picture(str(ruta_sede), width=Inches(6))

# Hallazgos
doc.add_heading('3. Hallazgos Principales', level=2)
for i, h in enumerate(datos_template['hallazgos'], 1):
    doc.add_heading(f'{i}. {h["titulo"]}', level=3)
    doc.add_paragraph(f'Evidencia: {h["evidencia"]}')
    doc.add_paragraph(f'Interpretación: {h["interpretacion"]}')
    doc.add_paragraph(f'Impacto: {h["impacto"]}')
    doc.add_paragraph(f'Decisión: {h["decision"]}')

# Recomendaciones
doc.add_heading('4. Recomendaciones', level=2)
tabla_rec = doc.add_table(rows=1, cols=5)
tabla_rec.style = 'Light Grid Accent 1'
headers = ['#', 'Acción', 'Prioridad', 'Responsable', 'Plazo']
for i, h in enumerate(headers):
    tabla_rec.rows[0].cells[i].text = h

for i, r in enumerate(datos_template['recomendaciones'], 1):
    row = tabla_rec.add_row()
    row.cells[0].text = str(i)
    row.cells[1].text = r['accion']
    row.cells[2].text = r['prioridad']
    row.cells[3].text = r['responsable']
    row.cells[4].text = r['plazo']

# Limitaciones
doc.add_heading('5. Limitaciones', level=2)
for l in datos_template['limitaciones']:
    doc.add_paragraph(l, style='List Bullet')

# Guardar
ruta_docx = BASE / 'exportaciones' / 'informe_smart_campus_v1.0.docx'
doc.save(str(ruta_docx))
print(f"DOCX generado: {ruta_docx.name}")
print(f"📏 Tamaño: {ruta_docx.stat().st_size / 1024:.1f} KB")

files.download(str(ruta_docx))
print("DOCX descargado")

DOCX generado: informe_smart_campus_v1.0.docx
📏 Tamaño: 243.2 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DOCX descargado


In [16]:
# METADATA DE TRAZABILIDAD
metadata = {
    'version': '1.0',
    'fecha_generacion': datetime.now().isoformat(),
    'periodo_analizado': kpis['periodo'],
    'fuentes_datos': [
        'dataset_integrado_Smart_Campus.csv',
        'dataset_limpio_Smart_Campus.csv',
        'dataset_publicable_Smart_Campus.csv',
        'smartcampus_nacional_arcgis.csv'
    ],
    'filtros_aplicados': {
        'periodo': kpis['periodo'],
        'facultades': 'todas',
        'recursos': 'todos',
        'sedes': 'todas'
    },
    'kpis': {
        'total_registros': int(kpis['total_registros']),
        'estudiantes_totales': int(kpis['estudiantes_totales']),
        'horas_estudiante': float(kpis['horas_estudiante']),
        'ocupacion_promedio': float(kpis['ocupacion_promedio'])
    },
    'archivos_generados': [
        'informe_smart_campus_v1.0.pdf',
        'informe_smart_campus_v1.0.docx',
        'informe_smart_campus.html'
    ],
    'generado_por': 'pipeline_reporteria_smart_campus.py'
}

ruta_metadata = BASE / 'exportaciones' / 'metadata_informe.json'
ruta_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print("Metadata de trazabilidad guardada:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

# CREAR GITHUB ACTIONS WORKFLOW
workflow = """
name: Generar Informe Smart Campus Automático

on:
  schedule:
    - cron: '0 8 * * 1'  # Todos los lunes a las 8 AM
  workflow_dispatch:

jobs:
  generar-informe:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: '3.10'
      - name: Instalar dependencias
        run: pip install pandas matplotlib jinja2 weasyprint python-docx
      - name: Generar informe
        run: python scripts/generar_informe.py
      - name: Subir artefactos
        uses: actions/upload-artifact@v3
        with:
          name: informe-smart-campus-generado
          path: exportaciones/
"""

ruta_workflow = BASE / '.github' / 'workflows'
ruta_workflow.mkdir(parents=True, exist_ok=True)
(ruta_workflow / 'generar_informe_smart_campus.yml').write_text(workflow, encoding='utf-8')

print("Workflow creado")
print(f"Ruta: {ruta_workflow / 'generar_informe_smart_campus.yml'}")

# VER TODOS LOS ARCHIVOS
print("TODOS LOS ARCHIVOS DE proyecto_smart_campus/:\n")
!ls -la /content/proyecto_smart_campus/

print("\nCONTENIDO DE LA CARPETA .github/workflows/:\n")
!ls -la /content/proyecto_smart_campus/.github/workflows/

# DESCARGAR EL WORKFLOW
from google.colab import files
files.download(str(ruta_workflow / 'generar_informe_smart_campus.yml'))
print("Workflow descargado")

Metadata de trazabilidad guardada:
{
  "version": "1.0",
  "fecha_generacion": "2026-11-17T23:13:44.773199",
  "periodo_analizado": "15/01/2026 - 14/07/2026",
  "fuentes_datos": [
    "dataset_integrado_Smart_Campus.csv",
    "dataset_limpio_Smart_Campus.csv",
    "dataset_publicable_Smart_Campus.csv",
    "smartcampus_nacional_arcgis.csv"
  ],
  "filtros_aplicados": {
    "periodo": "15/01/2026 - 14/07/2026",
    "facultades": "todas",
    "recursos": "todos",
    "sedes": "todas"
  },
  "kpis": {
    "total_registros": 10000,
    "estudiantes_totales": 220643,
    "horas_estudiante": 525358.0,
    "ocupacion_promedio": 70.77816899999999
  },
  "archivos_generados": [
    "informe_smart_campus_v1.0.pdf",
    "informe_smart_campus_v1.0.docx",
    "informe_smart_campus.html"
  ],
  "generado_por": "pipeline_reporteria_smart_campus.py"
}
Workflow creado
Ruta: /content/proyecto_smart_campus/.github/workflows/generar_informe_smart_campus.yml
TODOS LOS ARCHIVOS DE proyecto_smart_campus/:

t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Workflow descargado
